# TransitStats: V5 end-stop model tuning

Searches XGBoost settings for the V5 end-stop predictor, using the same cleaning, features and chronological 80/20 split as
`ml/train_endstop.py`. Report only: nothing here changes the live model.

**From the Mac (Colab CLI, preferred):** see `ml/lab/README.md` (Colab). The bundle is uploaded straight to the VM and
results are downloaded back; nothing goes to Drive.

**In the browser:** upload `ts-bundle.zip` to **My Drive/TransitStats/**, then Runtime → Run all. Results are saved next to the bundle.

Trip data is personal: don't share this notebook with outputs.


In [ ]:
import os, zipfile, sys
WORK = '/content/ts'
if os.path.exists('/content/ts-bundle.zip'):   # uploaded by `colab upload`
    BUNDLE, OUT = '/content/ts-bundle.zip', '/content/v5_tuning_results.csv'
else:                                           # browser run: bundle lives in Drive
    from google.colab import drive
    drive.mount('/content/drive')
    BUNDLE = '/content/drive/MyDrive/TransitStats/ts-bundle.zip'
    OUT = '/content/drive/MyDrive/TransitStats/v5_tuning_results.csv'
os.makedirs(WORK, exist_ok=True)
zipfile.ZipFile(BUNDLE).extractall(WORK)
os.chdir(WORK + '/ml'); sys.path.insert(0, WORK + '/ml')
!pip -q install firebase-admin


In [ ]:
import json, time, itertools
import pandas as pd, numpy as np
import train_endstop as t
from route_normalization import load_policies
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import top_k_accuracy_score
from xgboost import XGBClassifier

load_policies()
t.CSV_PATH = 'trips.csv'
df = t.clean(t.load_data(), json.load(open('stops.json')))
weights = t.compute_ride_count_weights(df)
X = t.build_features(df); y = df['end_stop_key']
split = max(1, int(len(df) * 0.2))
order = df.sort_values('start_time').index
tr, te = order[:-split], order[-split:]
w = pd.Series(weights, index=df.index)
classes = sorted(y.loc[tr].unique())
seen = y.loc[te].isin(classes).values
X_tr, y_tr, w_tr = X.loc[tr].values, y.loc[tr], w.loc[tr].values
X_te, y_te = X.loc[te].values[seen], y.loc[te][seen]
le = LabelEncoder().fit(classes)
y_tr_enc, y_te_enc = le.transform(y_tr), le.transform(y_te)
print(f'{len(df)} trips, {X.shape[1]} features, {len(classes)} end stops, {len(y_te)} test trips')

In [ ]:
try:
    import subprocess; subprocess.run(['nvidia-smi'], check=True, capture_output=True); DEVICE = 'cuda'
except Exception:
    DEVICE = 'cpu'
print('Running on', DEVICE)

def score(params):
    m = XGBClassifier(**params, eval_metric='mlogloss', verbosity=0, random_state=42,
                      tree_method='hist', device=DEVICE)
    t0 = time.time(); m.fit(X_tr, y_tr_enc, sample_weight=w_tr)
    p = m.predict_proba(X_te); labels = list(m.classes_)
    mask = np.isin(y_te_enc, labels)
    return dict(**params,
                top1=top_k_accuracy_score(y_te_enc[mask], p[mask], k=1, labels=labels),
                top3=top_k_accuracy_score(y_te_enc[mask], p[mask], k=min(3, len(labels)), labels=labels),
                seconds=round(time.time() - t0, 1))

BASELINE = dict(t.V5_PARAMS, subsample=1.0, colsample_bytree=1.0, min_child_weight=1)
GRID = dict(n_estimators=[100, 200, 400, 800], max_depth=[3, 4, 6, 8], learning_rate=[0.03, 0.1, 0.3],
            subsample=[0.8, 1.0], colsample_bytree=[0.6, 1.0], min_child_weight=[1, 3])
combos = [dict(zip(GRID, v)) for v in itertools.product(*GRID.values())]
print(len(combos), 'settings to try')

In [ ]:
base = score(BASELINE); print('Baseline (live V5 settings):', f"top-1 {base['top1']:.1%}  top-3 {base['top3']:.1%}")
rows = []
for i, c in enumerate(combos, 1):
    rows.append(score(c))
    if i % 25 == 0: print(f'{i}/{len(combos)} done')
res = pd.DataFrame(rows).sort_values(['top1', 'top3'], ascending=False)
res['top1_vs_baseline'] = (res.top1 - base['top1']).round(4)
res.to_csv(OUT, index=False); print('Saved', OUT)
res.head(15)

## Reading the result

A gain under about 1 point top-1 is likely noise on a test set this size. A bigger gain is a lead, not a
promotion: per `ml/STATUS.md`, changing the live model needs a dated report in `ml/evaluations/` and a
shadow comparison against the live path.